In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import csv
import math
import statistics
import pickle
from itertools import combinations
import igraph as ig

In [ ]:
data_s2 = pd.read_csv("../data/science.add9330_data_s2.csv", header=0)

In [ ]:
data_s2

In [ ]:
coord = pd.read_csv("../emisferi/larva/csv/Coordinate/Coordinate 2.0.csv", header=0)
coord

In [ ]:
coordinate_tre_colonne = pd.read_csv("../emisferi/larva/csv/Coordinate/Coordinate 2.0.csv", header=0)

coordinate_tre_colonne['x'] = coordinate_tre_colonne['x'].apply(lambda x: x/10)
coordinate_tre_colonne['y'] = coordinate_tre_colonne['y'].apply(lambda y: y/10)
coordinate_tre_colonne['z'] = coordinate_tre_colonne['z'].apply(lambda z: z/10)

coordinate_tre_colonne.to_csv('Coordinate.csv', index=False)

In [ ]:
coord = pd.read_csv("Coordinate.csv", header=0)

In [ ]:
coord

In [ ]:
adj_matrix = pd.read_csv("../data/all-all_connectivity_matrix.csv", header=[0], index_col=[0])
adj_matrix = adj_matrix.map(lambda x: 1 if x != 0 else 0)

In [ ]:
adj_matrix

In [ ]:
adj_matrix_a_a = pd.read_csv("../data/aa_connectivity_matrix.csv", header=[0], index_col=[0])
adj_matrix_a_a = adj_matrix_a_a.map(lambda x: 1 if x != 0 else 0)

aa_result = adj_matrix_a_a.stack()  
ones_aa = aa_result[aa_result == 1].index  

aa_list = [(int(row), int(col)) for row, col in ones_aa]
aa_dict = {edge : 'aa' for edge in aa_list}

In [ ]:
adj_matrix_a_d = pd.read_csv("../data/ad_connectivity_matrix.csv", header=[0], index_col=[0])
adj_matrix_a_d = adj_matrix_a_d.map(lambda x: 1 if x != 0 else 0)

ad_result = adj_matrix_a_d.stack()  
ones_ad = ad_result[ad_result == 1].index  

ad_list = [(int(row), int(col)) for row, col in ones_ad]
ad_dict = {edge : 'ad' for edge in ad_list}

In [ ]:
adj_matrix_d_a = pd.read_csv("../data/da_connectivity_matrix.csv", header=[0], index_col=[0])
adj_matrix_d_a = adj_matrix_d_a.map(lambda x: 1 if x != 0 else 0)

da_result = adj_matrix_d_a.stack()  
ones_da = da_result[da_result == 1].index  

da_list = [(int(row), int(col)) for row, col in ones_da]
da_dict = {edge : 'da' for edge in da_list}

In [ ]:
adj_matrix_d_d = pd.read_csv("../data/dd_connectivity_matrix.csv", header=[0], index_col=[0])
adj_matrix_d_d = adj_matrix_d_d.map(lambda x: 1 if x != 0 else 0)

dd_result = adj_matrix_d_d.stack()
ones_dd = aa_result[dd_result == 1].index 

dd_list = [(int(row), int(col)) for row, col in ones_dd]
dd_dict = {edge : 'dd' for edge in dd_list}

In [ ]:
len(aa_dict)+len(ad_dict)+len(da_dict)+len(dd_dict)

In [ ]:
all_connections = {**aa_dict, **ad_dict, **da_dict, **dd_dict}

In [ ]:
len(all_connections)

In [ ]:
edges = [(i, j) for i in adj_matrix.index for j in adj_matrix.columns if adj_matrix.loc[i, j] != 0]

In [ ]:
graph = nx.DiGraph(np.array(adj_matrix))

In [ ]:
G = nx.relabel_nodes(graph, dict(enumerate(adj_matrix.index)))

In [ ]:
G.number_of_edges()

In [ ]:
for u, v, data in G.edges(data=True):
    p1 = coord.loc[coord['id'] == u, ['x', 'y', 'z']].values.flatten().tolist()
    p2 = coord.loc[coord['id'] == v, ['x', 'y', 'z']].values.flatten().tolist()
    distanza = math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2 + (p2[2] - p1[2])**2)
    data['weight'] = round(distanza, 3)

    
    # data['type'] = all_connections[(int(u), int(v))]
    data['aa'] = aa_dict.get((int(u), int(v)), "")
    data['ad'] = ad_dict.get((int(u), int(v)), "")
    data['da'] = da_dict.get((int(u), int(v)), "")
    data['dd'] = dd_dict.get((int(u), int(v)), "")

In [ ]:
pesi = []

for u, v, data in G.edges(data=True):
    pesi.append(data['weight'])

max(pesi)

In [ ]:
for n, data in G.nodes(data=True):
    found = False
    try:
        data['celltype'] = data_s2[data_s2['left_id'] == str(n)]['celltype'].iloc[0]
        data['additional_annotations'] = data_s2[data_s2['left_id'] == str(n)]['additional_annotations'].iloc[0]
        data['level_7_cluster'] = data_s2[data_s2['left_id'] == str(n)]['level_7_cluster'].iloc[0]
        data['homolog'] = data_s2[data_s2['left_id'] == str(n)]['right_id'].iloc[0]
        data['hemisphere'] = 'right'
        found = True
    except IndexError:
        pass
    if not found:
        try:
            data['celltype'] = data_s2[data_s2['right_id'] == str(n)]['celltype'].iloc[0]
            data['additional_annotations'] = data_s2[data_s2['right_id'] == str(n)]['additional_annotations'].iloc[0]
            data['level_7_cluster'] = data_s2[data_s2['right_id'] == str(n)]['level_7_cluster'].iloc[0]
            data['homolog'] = data_s2[data_s2['right_id'] == str(n)]['left_id'].iloc[0]
            data['hemisphere'] = 'left'
        except IndexError:
            data['celltype'] = 'undefined'
            data['additional_annotations'] = 'undefined'
            data['level_7_cluster'] = 'undefined'
            data['hemisphere'] = 'undefined'
    node_coord = coord[coord['id']==n].iloc[0]
    data['x'] = node_coord['x']
    data['y'] = node_coord['y']
    data['z'] = node_coord['z']

In [ ]:
G.number_of_nodes()

In [ ]:
for e in G.edges(data=True):
    print(e)
    break

In [ ]:
for e in G.nodes(data=True):
    print(e)
    break

In [ ]:
nx.write_graphml(G, "weightedGraph_new_version.graphml")

In [ ]:
G = nx.read_graphml("weightedGraph_new_version.graphml")

In [ ]:
# G = nx.read_graphml("weightedGraph.graphml")

## Media delle distanze per area

Partendo dalla matrice di adiacenza pesata, calcolare la media delle distanze per area. 
Questa media va calcolata sommando i pesi degli archi uscenti dai nodi diviso il numero degli stessi archi.
Essendo la rete diretta, nel caso in cui fossero presenti due archi tra gli stessi nodi, ad esempio A->B e B->A, consideriamo questa distanza due volte.


In [ ]:
celltypes = list({attr.get('celltype', 'unknown') for _, attr in G.nodes(data=True)})
celltypes.remove('undefined')

In [ ]:
dist_info_celltypes = {}

for celltype in celltypes:
    nodes_celltype = [nodo for nodo, attr in G.nodes(data=True) if attr.get('celltype') == celltype]
    edges = list(G.edges())
    edges_celltype = []
    edges_weight = []

    for (s, t) in edges:
        if s in nodes_celltype:
            edges_celltype.append((s,t))
            edge_data = G.get_edge_data(s, t)
            edges_weight.append(edge_data['weight'])

    celltype_info = {
        'edges': edges_celltype,
        'edges_weight': edges_weight,
        'min': min(edges_weight),
        'max': max(edges_weight),
        'average': sum(edges_weight)/len(edges_weight),
        'median': statistics.median(edges_weight)
    }
    dist_info_celltypes[celltype] = celltype_info
        
with open('Larva/dist_info_celltypes.pickle', 'wb') as f:
    pickle.dump(dist_info_celltypes, f)

In [ ]:
with open('Larva/dist_info_celltypes.pickle', 'rb') as f:
    dist_info_superclasses = pickle.load(f)

In [ ]:
for celltype, info in dist_info_superclasses.items():
    print(celltype + ' average distance: ' + str(round(info['average'],4)))

A questo punto trovo i potenziali colonist per ogni area in questo modo: quei nodi che hanno almeno un arco uscente la cui distanza è maggiore della distanza media dell'area appena calcolata.


In [ ]:
with open('Larva/dist_info_celltypes.pickle', 'rb') as f:
    dist_info_superclasses = pickle.load(f)

In [ ]:
# passo 0.05 da 2 a 2.5
average_grid_search = [round(2+(x*0.05), 2) for x in range(0, 11)]
average_grid_search

In [ ]:
p_iteration = {}

for p in average_grid_search:
    print(p)
    p_iteration[p] = {}
    for celltype in celltypes:
        info_celltype = dist_info_celltypes[celltype]
        nodes_celltype = [nodo for nodo, attr in G.nodes(data=True) if attr.get('celltype') == celltype]
    
        candidate_colonists = set()
    
        for source in nodes_celltype:
            out_edges = list(G.out_edges(source, data=True))
            for edge in out_edges:
                if edge[2]['weight'] > p*info_celltype['average']:
                    candidate_colonists.add(source)
                    break
        with open('Larva/candidate_colonists_' + celltype + '_' + str(p) + '.pickle', 'wb') as f:
            pickle.dump(candidate_colonists, f)
        # print(celltype + ': ' + str(len(candidate_colonists)))

        p_iteration[p][celltype] = len(candidate_colonists)
    p_iteration[p]['tot'] = sum(p_iteration[p].values())
    print(p_iteration[p]['tot'])

In [ ]:
intestazioni_righe = list(next(iter(p_iteration.values())).keys())

with open('Larva/grid_search.csv', mode="w", newline="", encoding="utf-8") as file_csv:
    writer = csv.writer(file_csv)

    writer.writerow([""] + list(p_iteration.keys()))

    for riga in intestazioni_righe:
        writer.writerow([riga] + [p_iteration[colonna][riga] for colonna in p_iteration])


## Numero di target per ogni colonist candidato

In [ ]:
for p in average_grid_search:
    print(p)
    target_nodes_dict = {}
    num_target_nodes = {}
    for celltype in celltypes:
        with open('Larva/dist_info_celltypes.pickle', 'rb') as f:
            dist_info_celltypes = pickle.load(f)
        info_celltype = dist_info_celltypes[celltype]
        
        with open('Larva/candidate_colonists_' + celltype + '_' + str(p) + '.pickle', 'rb') as f:
            candidate_colonists = pickle.load(f)
    
        for source in candidate_colonists:
            out_edges = list(G.out_edges(source, data=True))
            target_nodes = set()
            for edge in out_edges:
                if edge[2]['weight'] > p*info_celltype['average']:
                    target_nodes.add(edge[1])
            if len(target_nodes)>1:
                target_nodes_dict[source] = list(target_nodes)
            num_target_nodes[source] = len(target_nodes)
            
    with open('Larva/candidate_colonists_targets' + str(p) + '.pickle', 'wb') as f:
                pickle.dump(target_nodes_dict, f)

In [ ]:
with open('Larva/num_target_nodes.csv', mode="w", newline="", encoding="utf-8") as file_csv:
    writer = csv.writer(file_csv)

    writer.writerow(["Node", "Num target nodes"])

    for key, value in num_target_nodes.items():
        writer.writerow([key] + [value])

## Distanza media della rete

In [ ]:
tot_dist = list(nx.get_edge_attributes(G, 'weight').values())
avg_dist = sum(tot_dist) / len(tot_dist)

In [ ]:
avg_dist

## Dai colonist potenziali ai colonist effettivi

Prendo un potenziale colonist, si considerano i suoi target e si costruisce il grafo completamente connesso i cui archi sono pesati dalla distanza. Si rimuovono tutti gli archi il cui peso è maggiore della distanza media dell'intero grafo. Sul grafo così ottenuto si calcolano le componenti connesse. Se c'è almeno una componente connessa di dimensione >= 2, il nodo è colonist.  Ogni componente è una colonia e va valutata.

In [ ]:
for p in average_grid_search:
    print(p)
    p_iteration[p] = {}
    verified_colonists = {}

    with open('Larva/candidate_colonists_targets' + str(p) + '.pickle', 'rb') as f:
            target_nodes_dict = pickle.load(f)
    
    for candidate, targets in target_nodes_dict.items():
        targets_info = {nodo: G.nodes[nodo] for nodo in targets}
        
        subgraph = nx.Graph()
        subgraph.add_nodes_from(list(targets))
        
        edges = list(combinations(list(targets), 2))
        subgraph.add_edges_from(edges)
        w = {}
        for source, target, data in subgraph.edges(data=True):
            source_x = targets_info[source]['x']
            source_y = targets_info[source]['y']
            source_z = targets_info[source]['z']
            target_x = targets_info[target]['x']
            target_y = targets_info[target]['y']
            target_z = targets_info[target]['z']
            distanza = math.sqrt((source_x - target_x)**2 + (source_y - target_y)**2 + (source_z - target_z)**2)
            data['weight'] = distanza
        to_remove = []
        for source, target, data in subgraph.edges(data=True):
            if data['weight']>avg_dist:
                to_remove.append((source, target))
        
        subgraph.remove_edges_from(to_remove)
    
        prov_connected_components = list(nx.connected_components(subgraph))
        connected_components = []
        
        for connected_component in prov_connected_components:
            if len(connected_component) > 1:
                connected_components.append(connected_component)
    
        if len(connected_components) >= 1:
            verified_colonists[candidate] = connected_components

    with open('Larva/colonists_' + str(p) + '.pickle', 'wb') as f:
            pickle.dump(verified_colonists, f)

In [ ]:
len(verified_colonists.keys())

## Dai colonist potenziali ai colonist

Prendo un potenziale colonist, prendo tutti i nodi target di archi uscenti dal potenziale colonist (dopo che ho applicato il filtro sulle connessioni), faccio la media delle distanze tra tutti i nodi target così selezionati

In [ ]:
for celltype in celltypes:
    info_celltype = dist_info_celltypes[celltype]
    nodes_celltype = [nodo for nodo, attr in G.nodes(data=True) if attr.get('celltype') == celltype]

    with open('Larva/candidate_colonists_' + celltype + '.pickle', 'rb') as f:
        candidate_colonists = pickle.load(f)

    colonists = set()

    for candidate in candidate_colonists:
        out_edges = list(G.out_edges(str(candidate), data=True))
        nodes_to_verify = []
        for edge in out_edges:
            if edge[2]['weight'] > info_celltype['average']:
                nodes_to_verify.append(edge[1])
    
    with open('Larva/colonists_' + celltype + '.pickle', 'wb') as f:
        pickle.dump(colonists, f)
    print(celltype + ': ' + str(len(colonists)))

In [ ]:
edges = list(G.edges(data=True))

with open('Data/edges.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    # Scrivi l'intestazione delle colonne
    writer.writerow(['source', 'target', 'weight'])
    
    # Scrivi i dati riga per riga
    for source, target, attributes in edges:
        writer.writerow([source, target, attributes['weight']])

In [ ]:
list(G.nodes(data=True))[0]

In [ ]:
nodes = list(G.nodes(data=True))

with open('Data/nodes.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    # Scrivi l'intestazione delle colonne
    writer.writerow(['id', 'celltype', 'additional_annotations', 'level_7_cluster', 'x', 'y', 'z', 'hemisphere'])
    
    # Scrivi i dati riga per riga
    for node, attributes in nodes:
        writer.writerow([node, attributes['celltype'], attributes['additional_annotations'], attributes['level_7_cluster'], 
                         attributes['x'], attributes['y'], attributes['z'], attributes['hemisphere']])

# Analisi iniziali

In [ ]:
sum(dict(G.degree()).values()) / G.number_of_nodes()

In [ ]:
largest = max(nx.strongly_connected_components(G), key=len)
largest_scc = G.subgraph(list(largest))

In [ ]:
nx.average_shortest_path_length(largest_scc)

In [ ]:
nx.average_shortest_path_length(largest_scc, weight='weight')

In [ ]:
degree_connectivity = list(nx.average_degree_connectivity(G).values())
sum(degree_connectivity)/len(degree_connectivity)

In [ ]:
degree_connectivity = list(nx.average_degree_connectivity(G, weight='weight').values())
sum(degree_connectivity)/len(degree_connectivity)

In [ ]:
G.nodes['15728017']['x']

In [ ]:
pesi = []

for u, v, data in G.edges(data=True):
    pesi.append(data['weight'])

max(pesi)

In [ ]:
G_igraph = ig.Graph.from_networkx(G)

In [ ]:
components = G_igraph.connected_components()  
largest_component = components.giant()

In [ ]:
G_igraph.average_path_length()

In [ ]:
largest_component.average_path_length()

In [ ]:
G_igraph.average_path_length(weights='weight')

In [ ]:
largest_component.average_path_length(weights='weight')

In [ ]:
G_igraph.average_path_length(directed=True, unconn=False) 

In [ ]:
largest_component.average_path_length(directed=True, unconn=False) 

In [ ]:
G_igraph.average_path_length(weights='weight', directed=True, unconn=False)

In [ ]:
G_igraph.diameter()

In [ ]:
in_degrees = [deg for _, deg in G.in_degree()]  # Lista degli in-degree per ciascun nodo
average_in_degree = sum(in_degrees) / len(G.nodes())

print(f"Average in-degree: {average_in_degree}")

In [ ]:
out_degrees = [deg for _, deg in G.out_degree()]  # Lista degli out-degree per ciascun nodo
average_out_degree = sum(out_degrees) / len(G.nodes())

print(f"Average in-degree: {average_out_degree}")

In [ ]:
# Calcolo dell'average weighted in-degree
weighted_in_degrees = [deg for _, deg in G.in_degree(weight='weight')]  # In-degree pesato
average_weighted_in_degree = sum(weighted_in_degrees) / len(G.nodes())

print(f"Average Weighted In-Degree: {average_weighted_in_degree}")

In [ ]:
# Calcolo dell'average weighted in-degree
weighted_out_degrees = [deg for _, deg in G.out_degree(weight='weight')]  # Out-degree pesato
average_weighted_out_degree = sum(weighted_out_degrees) / len(G.nodes())

print(f"Average Weighted Out-Degree: {average_weighted_out_degree}")

In [ ]:
hemispheres_list = [attr['hemisphere'] for _, attr in G.nodes(data=True) if 'hemisphere' in attr]

In [ ]:
from collections import Counter

# Conta le occorrenze
occurrences = Counter(hemispheres_list)

print(occurrences)